In [1]:
!pip install pandas numpy matplotlib seaborn tqdm scikit-learn datasets
!pip install emoji
!pip uninstall nltk -y
!pip install nltk
!pip install spacy
!python -m spacy download en_core_web_sm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.4/487.4 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 590.6/590.6 kB 8.0 MB/s eta 0:00:00
Found existing installation: nltk 3.9.1
Uninstalling nltk-3.9.1:
  Successfully uninstalled nltk-3.9.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 71.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# 📌 File and Directory Management
import os
import tarfile

# 📌 Data Manipulation
import pandas as pd
import numpy as np

# 📌 Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# 📌 Text Processing (NLP)
import re
import emoji
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob
from collections import Counter
import itertools

# 📌 Machine Learning and Utilities
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# 📌 Pandas Display Settings
pd.set_option("display.max_colwidth", None)  # Show entire text columns


# Open zipped file

In [26]:
import os
import pandas as pd
from tqdm import tqdm

dataset_path = "aclImdb"

def load_imdb_train_data(directory):
    data = []
    labels = []
    split = "train"  # train
    for sentiment, label in [("pos", 1), ("neg", 0)]:
        path = os.path.join(directory, split, sentiment)
        for filename in tqdm(os.listdir(path), desc=f"Loading {split}/{sentiment}"):
            with open(os.path.join(path, filename), "r", encoding="utf-8") as file:
                data.append(file.read())
                labels.append(label)
    return pd.DataFrame({"review": data, "sentiment": labels})


df_train = load_imdb_train_data(dataset_path)
print(f"Tamanho do conjunto de treino: {len(df_train)}")

Loading train/neg: 100%|██████████| 12500/12500 [00:00<00:00, 35032.90it/s]

Tamanho do conjunto de treino: 25000


# Removing HTML Tags and Special Characters

In [27]:
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# NLTK Downloads
nltk.download("stopwords")
nltk.download("punkt")
nltk.download('punkt_tab')
# Cleaning function
def clean_text(text):
    text = re.sub(r"<.*?>", "", text)  # Remove HTML tags
    text = re.sub(r"[^a-zA-Z\s]", "", text)  # Remove special characters and numbers
    text = text.lower()  # Convert to lowercase
    return text

# Stopwords
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word.lower() not in stop_words]
    return " ".join(tokens)

# Apply preprocessing only in training
df_train["review"] = df_train["review"].apply(clean_text)
df_train["clean_review"] = df_train["review"].apply(remove_stopwords)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [28]:
# Check and remove duplicates
duplicatas = df_train.duplicated(subset=['clean_review']).sum()
print(f"📌 Duplicates in training: {duplicatas}")
df_train = df_train.drop_duplicates(subset=['clean_review'], keep='first')
print(f"📌 Workout size after duplicates: {len(df_train)}")

# Validation
print("Clean example:", df_train['clean_review'].iloc[0])
df_train['clean_length'] = df_train['clean_review'].apply(lambda x: len(x.split()))
print(f"📌 Average length in training: {df_train['clean_length'].mean()}")

# Save only the workout
df_train[['clean_review', 'sentiment']].to_csv('imdb_train_preprocessed.csv', index=False)
print("📌 Workout saved in 'imdb_train_preprocessed.csv'")

📌 Duplicates in training: 98
📌 Workout size after duplicates: 24902
Clean example: robin williams best combine comedy pathos comes bit shrill donald moffat onenote fatherinlaw jeff bridges excellent though quarterback holly palance pamela reed marvelous carrying film rough spots fills time nicely little
📌 Average length in training: 120.06634005300779
📌 Workout saved in 'imdb_train_preprocessed.csv'


# Validation

In [29]:
# Check for empty or very short reviews in training
df_train['is_empty'] = df_train['clean_review'].apply(lambda x: len(str(x).strip()) == 0)
df_train['is_too_short'] = df_train['clean_review'].apply(lambda x: len(str(x).split()) < 3)

print("📌 Training Validation:")
print(f"📌 Empty reviews: {df_train['is_empty'].sum()}")
print(f"📌 Very short reviews (< 3 words): {df_train['is_too_short'].sum()}")

# Remove empty reviews
df_train = df_train[~df_train['is_empty']]
print(f"📌 Training size after removing voids: {len(df_train)}")


# Check average and maximum review length
df_train['review_length'] = df_train['clean_review'].apply(lambda x: len(str(x).split()))
print(f"📌 Average length of reviews (training): {df_train['review_length'].mean()}")
print(f"📌 Maximum length of reviews (training): {df_train['review_length'].max()}")

# Check for unwanted leftovers (e.g. HTML)
df_train['has_html'] = df_train['clean_review'].apply(lambda x: bool(re.search(r"<.*?>", str(x))))
print(f"📌 Reviews with residual HTML (training): {df_train['has_html'].sum()}")

📌 Training Validation:
📌 Empty reviews: 0
📌 Very short reviews (< 3 words): 0
📌 Training size after removing voids: 24902
📌 Average length of reviews (training): 120.06634005300779
📌 Maximum length of reviews (training): 1420
📌 Reviews with residual HTML (training): 0
